In [ ]:
from trustutils import run
run.TRUST_parameters()
run.introduction('MMS Diffusion 3D', '2026')


In [ ]:
run.reset()

NB_PROCS = 2

case = run.addCase('.', 'MMS_Diff3D.data', nbProcs=NB_PROCS)
case.partition()          # trust -partition MMS_Diff3D.data 2 -> PAR_MMS_Diff3D.data + DOM_000*.Zones
# --- mesh-convergence scan (shared ladder: y cells x2 / x4, x = z frozen) ----
# The eigenmode sin(pi y) is 1D in y, so only the active y direction is
# refined; the passive node counts keep their baseline values (the dataset
# already sits at 21^3: at coarser meshes the P0 projection excites higher
# discrete modes, see the form history). Levels are instantiated from
# src/MMS_Diff3D.tpl.data; each level runs in its own build/scan/<lvl>/
# directory so that the partitioned DOM_000*.Zones never collide.
NY0 = 21                              # baseline y node count (L0); x = z = 21
LEVELS = {
    "L1": {"nx": 21, "ny": 2*NY0-1, "nz": 21},   # y cells x2 -> h/2
    "L2": {"nx": 21, "ny": 4*NY0-3, "nz": 21},   # y cells x4 -> h/4
}
scan_cases = {}
for lvl, dic in LEVELS.items():
    c = run.addCaseFromTemplate("MMS_Diff3D.tpl.data", f"scan/{lvl}", dic=dic,
                                targetData=f"MMS_Diff3D_{lvl}.data",
                                nbProcs=NB_PROCS, excluNR=True)
    c.partition()
    scan_cases[lvl] = c
run.printCases()
run.runCases()
run.tablePerf()

In [ ]:
from trustutils.files import SonFile
import numpy as np

BUILD = run.BUILD_DIRECTORY

def read_son(case, name):
    """Return (time, values) for probe `name` of case `case` (dataset name)."""
    path = f"{BUILD}/{case}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

# MMS_Diff3D: b(x, y, 0) = sin(pi y) with Dirichlet zero at y = 0, 1 and
# Neumann walls elsewhere is an exact eigenmode of the diffusion operator:
#     b(x, y, t) = sin(pi y) exp(-eta pi^2 t),  eta = 0.05.
# All three probes sit on the mode, so each must follow the same analytic
# decay.  The expected ratio uses the ACTUAL final time of the run.
eta = 0.05

for name in ('SONDE_Y025', 'SONDE_Y050', 'SONDE_Y075'):
    t, b = read_son(case.dataFileName_, name)
    ratio = b[-1] / b[0]
    expected = np.exp(-eta * np.pi**2 * t[-1])
    err = abs(ratio - expected) / expected
    print(f"{name}: ratio={ratio:.4f} expected={expected:.4f} err={err:.3e}")
    assert err < 0.10, f"{name} decay error too large: {err:.3e}"
print("OK: sin(pi y) mode decays at the analytic eta pi^2 rate at all probes.")


## Mesh convergence

The shared refinement ladder, adapted to the 1D eigenmode physics: only the
active `y` direction is refined (`L0` = the baseline mesh `21×21×21` — the
dataset already sits at 21³ because coarser meshes excite higher discrete
P0 modes — `L1` = y cells ×2 (`21×41×21`), `L2` = y cells ×4 (`21×81×21`)).

The *Checks* ratio metric saturates at the implicit-Euler temporal error,
common to every mesh. To expose the **spatial** error, the scan compares
$b(t)/b(0)$ against the exact implicit-Euler decay built with the analytic
eigenvalue $\lambda = \eta\pi^2$ on the probe's own time grid: the time
scheme cancels out and the residual is the spatial one,
$\lambda_h - \lambda = O(h^2)$. The observed order is $\log_2(e_k/e_{k+1})$.

In [ ]:
SONDE_TAGS_SCAN = ["SONDE_Y025", "SONDE_Y050", "SONDE_Y075"]
ETA = 0.05                       # magnetic diffusivity of the dataset

def read_son_dir(subdir, case_name, name):
    """(time, values) of probe `name` of case `case_name` in a build subdir."""
    path = f"{BUILD}/{subdir}/{case_name}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def implicit_ref(t, lam):
    """Exact implicit-Euler decay ratio on the probe time grid (any dt sequence)."""
    t = np.asarray(t, dtype=float)
    ref = np.ones_like(t)
    for i in range(1, len(t)):
        ref[i:] *= 1.0 / (1.0 + lam * (t[i] - t[i-1]))
    return ref

LAM = ETA * np.pi**2                # analytic eigenvalue eta*pi^2
def spatial_ratio_error(subdir, case_name):
    """Max |b(t)/b(0) - implicit-Euler analytic decay| over the scan probes."""
    worst = 0.0
    for tag in SONDE_TAGS_SCAN:
        t, b = read_son_dir(subdir, case_name, tag)
        if b[0] == 0:
            continue
        worst = max(worst, float(np.max(np.abs(b / b[0] - implicit_ref(t, LAM)))))
    return worst

scan = {"L0": ("", case.dataFileName_),
        "L1": (f"scan/L1", scan_cases["L1"].dataFileName_),
        "L2": (f"scan/L2", scan_cases["L2"].dataFileName_)}
scan_nodes = {"L0": (21, 21, 21), "L1": (21, 41, 21), "L2": (21, 81, 21)}
scan_err = {lvl: spatial_ratio_error(sub, cn) for lvl, (sub, cn) in scan.items()}

print(f"level  nodes       h_y        |ratio - implicit ref|   observed order")
prev = None
for lvl in ("L0", "L1", "L2"):
    nx, ny, nz = scan_nodes[lvl]
    e = scan_err[lvl]
    order = "--"
    if prev is not None and prev > 0 and e > 0:
        order = f"{np.log2(prev / e):.2f}"
    print(f"{lvl:<6} {nx:>3}x{ny}x{nz:<3} {1.0/(ny-1):<10.4g} {e:<26.4e} {order}")
    prev = e

assert scan_err["L1"] < scan_err["L0"], f"no convergence L0 -> L1: {scan_err}"
assert scan_err["L2"] < scan_err["L1"], f"no convergence L1 -> L2: {scan_err}"
print("OK: spatial error decreases monotonically under refinement (order ~2).")